# 🚦 Day 1 — Data Pipeline + EDA
## Gridlock Hackathon 2.0 | Bengaluru Parking Violation Intelligence

**Goal:** Load the full 3L-row violation dataset, clean it, engineer core features,
and extract every insight needed to build the priority-scoring model on Day 2.

**Structure:**
1. Environment Setup
2. Data Loading & First Look
3. Data Cleaning
4. Feature Engineering
5. Exploratory Data Analysis (EDA)
6. Save Clean Data

> Run cells top-to-bottom. All heavy operations show a `tqdm` progress bar.


---
## 1. Environment Setup
Install and import everything needed for the full pipeline.
One-time cell — run once, then skip on re-runs.


In [ ]:
# Install dependencies (run once)
import subprocess, sys

packages = [
    "pandas", "numpy", "matplotlib", "seaborn",
    "folium", "tqdm", "pyarrow", "geopandas",
    "scikit-learn", "plotly", "h3", "branca"
]

print("Installing packages...")
for pkg in packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"],
        capture_output=True
    )
print("✅ All packages installed.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium
import warnings
import ast
import re
import json
from tqdm.notebook import tqdm
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
tqdm.pandas()  # enables df.progress_apply()

# ── Plotting style ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "figure.dpi": 120,
})
sns.set_palette("husl")

print("✅ Imports done.")


---
## 2. Data Loading & First Look

Load the CSV, print shape, dtypes, nulls, and sample rows.
Update `DATA_PATH` to point to your downloaded file.


In [ ]:
# ── UPDATE THIS PATH ──────────────────────────────────────────────────────
DATA_PATH = "jan_to_may_police_violation_anonymized.csv"   # <-- change if needed
# ──────────────────────────────────────────────────────────────────────────

print(f"Loading data from: {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"✅ Loaded  →  {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)


In [ ]:
# ── Schema overview ───────────────────────────────────────────────────────
print("=" * 60)
print("COLUMN TYPES")
print("=" * 60)
print(df_raw.dtypes.to_string())

print("\n" + "=" * 60)
print("NULL COUNTS & PERCENTAGE")
print("=" * 60)
null_df = pd.DataFrame({
    "null_count": df_raw.isnull().sum(),
    "null_pct":   (df_raw.isnull().mean() * 100).round(2)
}).sort_values("null_pct", ascending=False)
print(null_df[null_df.null_count > 0].to_string())


In [ ]:
# ── Basic stats for numeric columns ───────────────────────────────────────
df_raw[["latitude", "longitude"]].describe().round(4)


---
## 3. Data Cleaning

Steps:
- Drop exact duplicates
- Drop rows with missing lat/lon (can't map them)
- Parse datetime columns
- Normalise string columns (strip, upper-case)
- Parse JSON-like `violation_type` and `offence_code` arrays


In [ ]:
df = df_raw.copy()

# ── 3a. Drop exact duplicates ──────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Dropped {before - len(df):,} exact duplicates → {len(df):,} rows remain")

# ── 3b. Drop rows with no coordinates ─────────────────────────────────────
before = len(df)
df.dropna(subset=["latitude", "longitude"], inplace=True)
print(f"Dropped {before - len(df):,} rows missing lat/lon → {len(df):,} rows remain")

# ── 3c. Bengaluru bounding box filter (sanity check) ──────────────────────
# Bengaluru roughly: lat 12.7–13.2, lon 77.4–77.9
before = len(df)
df = df[
    df.latitude.between(12.7, 13.2) &
    df.longitude.between(77.4, 77.9)
]
print(f"Dropped {before - len(df):,} rows outside Bengaluru bounds → {len(df):,} rows remain")


In [ ]:
# ── 3d. Parse datetime columns ────────────────────────────────────────────
print("Parsing datetimes...")
for col in tqdm(["created_datetime", "closed_datetime", "modified_datetime",
                  "action_taken_timestamp", "data_sent_to_scita_timestamp",
                  "validation_timestamp"], desc="Datetime cols"):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

# Convert to IST (UTC+5:30)
IST = "Asia/Kolkata"
df["created_ist"] = df["created_datetime"].dt.tz_convert(IST)
print("✅ Datetime parsing done. Sample:")
df[["created_datetime", "created_ist"]].head(3)


In [ ]:
# ── 3e. Normalise string columns ──────────────────────────────────────────
str_cols = ["vehicle_type", "validation_status", "police_station",
            "junction_name", "data_sent_to_scita"]
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()

# Standardise NULLs written as string "NULL" or "NAN"
df.replace({"NULL": np.nan, "NAN": np.nan, "NONE": np.nan}, inplace=True)

print("✅ String normalisation done.")
print("\nVehicle type value counts:")
print(df["vehicle_type"].value_counts().head(15))


In [ ]:
# ── 3f. Parse violation_type and offence_code arrays ──────────────────────
# These columns look like: '["NO PARKING","WRONG PARKING"]'

def safe_parse_list(val):
    """Parse a JSON/Python list string into a Python list."""
    if pd.isna(val) or val in ("nan", "NULL", ""):
        return []
    try:
        parsed = json.loads(val)
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        try:
            return ast.literal_eval(str(val))
        except Exception:
            return []

print("Parsing violation_type column...")
df["violation_list"] = [safe_parse_list(v) for v in tqdm(
    df["violation_type"], desc="violation_type", total=len(df))]

print("Parsing offence_code column...")
df["offence_code_list"] = [safe_parse_list(v) for v in tqdm(
    df["offence_code"], desc="offence_code", total=len(df))]

df["num_violations"] = df["violation_list"].apply(len)
print(f"\n✅ Parsed. Average violations per record: {df['num_violations'].mean():.2f}")
print("\nTop violation types (all records):")
all_violations = [v for vlist in df["violation_list"] for v in vlist]
print(pd.Series(all_violations).value_counts())


---
## 4. Feature Engineering

Build every feature the priority model will need:
- **Temporal features:** hour, day of week, is_peak_hour, is_weekend
- **Severity score:** weighted sum of offence codes (worse offences = higher weight)
- **Junction flag:** violations near a named junction are higher impact
- **Road type proxy:** inferred from address string (Main Road > Cross > Layout)
- **Data quality flag:** sent to SCITA + validation status


In [ ]:
# ── 4a. Temporal features ─────────────────────────────────────────────────
print("Engineering temporal features...")
df["hour"]       = df["created_ist"].dt.hour
df["day_of_week"] = df["created_ist"].dt.dayofweek   # 0=Mon, 6=Sun
df["day_name"]   = df["created_ist"].dt.day_name()
df["month"]      = df["created_ist"].dt.month
df["month_name"] = df["created_ist"].dt.month_name()
df["week"]       = df["created_ist"].dt.isocalendar().week.astype(int)
df["date"]       = df["created_ist"].dt.date

# Peak hours: morning rush 7–10 AM, evening rush 5–9 PM
df["is_morning_peak"] = df["hour"].between(7, 10).astype(int)
df["is_evening_peak"] = df["hour"].between(17, 21).astype(int)
df["is_peak_hour"]    = (df["is_morning_peak"] | df["is_evening_peak"]).astype(int)
df["is_weekend"]      = (df["day_of_week"] >= 5).astype(int)

# Time bucket for heatmaps
def time_bucket(h):
    if 0  <= h <  6: return "Night (0–6)"
    if 6  <= h < 10: return "Morning Rush (6–10)"
    if 10 <= h < 14: return "Midday (10–14)"
    if 14 <= h < 18: return "Afternoon (14–18)"
    return "Evening Rush (18–24)"

df["time_bucket"] = df["hour"].apply(time_bucket)

print("✅ Temporal features done.")
df[["hour","day_name","is_peak_hour","time_bucket"]].head(4)


In [ ]:
# ── 4b. Severity score ────────────────────────────────────────────────────
# Rationale: some offences cause more congestion than others.
# Weights based on operational impact (higher = worse congestion caused).

SEVERITY_WEIGHTS = {
    109: 5,   # PARKING OPPOSITE TO ANOTHER PARKED VEHICLE  (blocks full lane)
    108: 5,   # DOUBLE PARKING
    107: 4,   # PARKING IN A MAIN ROAD
    111: 4,   # PARKING NEAR BUSTOP/SCHOOL/HOSPITAL
    104: 3,   # PARKING NEAR ROAD CROSSING / JUNCTION
    112: 2,   # WRONG PARKING (generic)
    113: 1,   # NO PARKING (lowest — sign violation, may not block)
}
DEFAULT_SEVERITY = 1  # unknown codes

def compute_severity(code_list):
    if not code_list:
        return 0
    return sum(SEVERITY_WEIGHTS.get(int(c), DEFAULT_SEVERITY) for c in code_list)

print("Computing severity scores...")
df["severity_score"] = [compute_severity(cl) for cl in tqdm(
    df["offence_code_list"], desc="severity", total=len(df))]

print(f"✅ Severity score done.")
print(df["severity_score"].describe().round(2))


In [ ]:
# ── 4c. Junction proximity flag ───────────────────────────────────────────
# junction_name is already in data. "No Junction" means not near a key junction.

df["near_junction"] = (~df["junction_name"].isin([np.nan, "NO JUNCTION"])).astype(int)
print("Near junction distribution:")
print(df["near_junction"].value_counts())
print(f"\n{df['near_junction'].mean()*100:.1f}% of violations are near a named junction")


In [ ]:
# ── 4d. Road type proxy from address string ───────────────────────────────
# Main Road > Cross Road > Layout/Colony (lane/road width proxy)

ROAD_RANK = {
    "MAIN ROAD": 4,
    "OUTER RING ROAD": 4,
    "RING ROAD": 4,
    "HIGHWAY": 4,
    "CROSS ROAD": 3,
    "CROSS": 3,
    "ROAD": 2,
    "AVENUE": 2,
    "STREET": 2,
    "LAYOUT": 1,
    "COLONY": 1,
    "NAGAR": 1,
}

def infer_road_rank(loc_str):
    if pd.isna(loc_str):
        return 1
    loc_upper = str(loc_str).upper()
    for keyword, rank in ROAD_RANK.items():
        if keyword in loc_upper:
            return rank
    return 1

print("Inferring road type from address...")
df["road_rank"] = [infer_road_rank(l) for l in tqdm(
    df["location"], desc="road_rank", total=len(df))]

print("\nRoad rank distribution:")
print(df["road_rank"].value_counts().sort_index())


In [ ]:
# ── 4e. Vehicle weight (congestion footprint) ─────────────────────────────
# Large vehicles blocking a lane cause more congestion than scooters

VEHICLE_WEIGHT = {
    "BUS": 5, "TANKER": 5, "TRUCK": 5, "LORRY": 5,
    "MAXI-CAB": 4, "MAXI CAB": 4,
    "PASSENGER AUTO": 3, "AUTO": 3,
    "CAR": 2, "TAXI": 2, "CAB": 2,
    "MOTOR CYCLE": 1, "SCOOTER": 1, "BICYCLE": 1,
}
DEFAULT_VEH = 2

df["vehicle_weight"] = df["vehicle_type"].map(
    lambda v: next((w for k, w in VEHICLE_WEIGHT.items() if str(v).upper().startswith(k)), DEFAULT_VEH)
)

print("Vehicle weight distribution:")
print(df.groupby("vehicle_type")["vehicle_weight"].first().sort_values(ascending=False))


In [ ]:
# ── 4f. Data quality / usability flags ───────────────────────────────────
df["sent_to_scita"]   = (df["data_sent_to_scita"] == "TRUE").astype(int)
df["is_approved"]     = (df["validation_status"] == "APPROVED").astype(int)
df["is_rejected"]     = (df["validation_status"] == "REJECTED").astype(int)
df["is_validated"]    = df["validation_status"].notna().astype(int)

print("Data quality summary:")
print(f"  Sent to SCITA:   {df['sent_to_scita'].sum():,} ({df['sent_to_scita'].mean()*100:.1f}%)")
print(f"  Approved:        {df['is_approved'].sum():,} ({df['is_approved'].mean()*100:.1f}%)")
print(f"  Rejected:        {df['is_rejected'].sum():,} ({df['is_rejected'].mean()*100:.1f}%)")
print(f"  Unvalidated:     {(~df['is_validated'].astype(bool)).sum():,}")


In [ ]:
# ── 4g. Composite impact score (per-record) ───────────────────────────────
# Used later for hex-cell aggregation
# severity × vehicle_size × road_importance × junction_multiplier × peak_multiplier

df["impact_score"] = (
    df["severity_score"] *
    df["vehicle_weight"] *
    df["road_rank"] *
    (1 + 0.5 * df["near_junction"]) *
    (1 + 0.3 * df["is_peak_hour"])
).round(2)

print("Impact score distribution:")
print(df["impact_score"].describe().round(2))
print(f"\nTop 5 impact score records:")
print(df.nlargest(5, "impact_score")[["location","vehicle_type","violation_list","impact_score"]])


---
## 5. Exploratory Data Analysis

Systematically visualise every dimension relevant to enforcement priority.


In [ ]:
# ── 5a. Violation type distribution ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
viol_counts = pd.Series(all_violations).value_counts()
bars = ax.barh(viol_counts.index[::-1], viol_counts.values[::-1], color=sns.color_palette("husl", len(viol_counts)))
ax.set_xlabel("Number of Records")
ax.set_title("Violation Type Distribution")
for bar, val in zip(bars, viol_counts.values[::-1]):
    ax.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("eda_01_violation_types.png", bbox_inches="tight")
plt.show()
print("Saved → eda_01_violation_types.png")


In [ ]:
# ── 5b. Vehicle type distribution ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
veh_counts = df["vehicle_type"].value_counts().head(15)
bars = ax.bar(veh_counts.index, veh_counts.values,
              color=sns.color_palette("Set2", len(veh_counts)))
ax.set_ylabel("Count")
ax.set_title("Violations by Vehicle Type (Top 15)")
ax.tick_params(axis="x", rotation=35)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"{int(bar.get_height()):,}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("eda_02_vehicle_types.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 5c. Hour-of-day distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All violations by hour
hour_counts = df["hour"].value_counts().sort_index()
axes[0].bar(hour_counts.index, hour_counts.values,
            color=["#e74c3c" if h in range(7,11) or h in range(17,22)
                   else "#3498db" for h in hour_counts.index])
axes[0].set_xlabel("Hour of Day (IST)")
axes[0].set_ylabel("Violation Count")
axes[0].set_title("Violations by Hour (Red = Peak Hours)")
axes[0].set_xticks(range(0, 24, 2))

# Peak vs non-peak pie
peak_counts = df["is_peak_hour"].value_counts()
axes[1].pie(peak_counts.values, labels=["Off-Peak", "Peak Hour"],
            colors=["#3498db", "#e74c3c"], autopct="%1.1f%%", startangle=90)
axes[1].set_title("Peak vs Off-Peak Violations")

plt.tight_layout()
plt.savefig("eda_03_hourly.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 5d. Day-of-week distribution ──────────────────────────────────────────
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
day_counts = df["day_name"].value_counts().reindex(day_order)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#e74c3c" if d in ["Saturday","Sunday"] else "#2ecc71" for d in day_order]
ax.bar(day_order, day_counts.values, color=colors)
ax.set_ylabel("Violation Count")
ax.set_title("Violations by Day of Week (Red = Weekend)")
ax.tick_params(axis="x", rotation=20)
for i, (bar, val) in enumerate(zip(ax.patches, day_counts.values)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f"{val:,}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("eda_04_dayofweek.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 5e. Month-wise trend ──────────────────────────────────────────────────
month_order = ["January","February","March","April","May"]
month_counts = df["month_name"].value_counts().reindex(month_order).dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(month_counts.index, month_counts.values, marker="o", linewidth=2.5,
        color="#9b59b6", markersize=8)
ax.fill_between(month_counts.index, month_counts.values, alpha=0.15, color="#9b59b6")
ax.set_ylabel("Violation Count")
ax.set_title("Monthly Violation Trend (Jan–May)")
for x, y in zip(month_counts.index, month_counts.values):
    ax.annotate(f"{y:,}", (x, y), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("eda_05_monthly.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 5f. Hour × Day-of-Week heatmap ───────────────────────────────────────
pivot = df.pivot_table(
    index="day_name", columns="hour", values="id", aggfunc="count"
).reindex(day_order)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, linewidths=0.3,
            cbar_kws={"label": "Violation Count"})
ax.set_title("Violation Heatmap: Hour of Day × Day of Week")
ax.set_xlabel("Hour (IST)")
ax.set_ylabel("Day")
plt.tight_layout()
plt.savefig("eda_06_hour_day_heatmap.png", bbox_inches="tight")
plt.show()
print("This heatmap directly tells you WHEN enforcement is most needed.")


In [ ]:
# ── 5g. Police station / zone analysis ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
station_counts = df["police_station"].value_counts().head(25)
bars = ax.barh(station_counts.index[::-1], station_counts.values[::-1],
               color=sns.color_palette("magma", 25))
ax.set_xlabel("Violation Count")
ax.set_title("Top 25 Police Stations by Violation Count")
for bar, val in zip(bars, station_counts.values[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("eda_07_police_stations.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 5h. Severity score distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df["severity_score"], bins=30, color="#e67e22", edgecolor="white")
axes[0].set_xlabel("Severity Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Severity Scores")

# Severity by vehicle type (top 10)
sev_veh = df.groupby("vehicle_type")["severity_score"].mean().sort_values(ascending=False).head(10)
axes[1].barh(sev_veh.index[::-1], sev_veh.values[::-1], color="#e74c3c")
axes[1].set_xlabel("Mean Severity Score")
axes[1].set_title("Mean Severity by Vehicle Type (Top 10)")

plt.tight_layout()
plt.savefig("eda_08_severity.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 5i. Junction analysis ─────────────────────────────────────────────────
top_junctions = (
    df[df["junction_name"].notna()]
    .groupby("junction_name")
    .agg(count=("id", "count"), avg_severity=("severity_score", "mean"))
    .sort_values("count", ascending=False)
    .head(20)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By count
bars = axes[0].barh(top_junctions.index[::-1], top_junctions["count"][::-1],
                     color="#1abc9c")
axes[0].set_title("Top 20 Junctions by Violation Count")
axes[0].set_xlabel("Count")

# By avg severity
bars2 = axes[1].barh(top_junctions.sort_values("avg_severity", ascending=False).index[::-1],
                      top_junctions.sort_values("avg_severity", ascending=False)["avg_severity"][::-1],
                      color="#e74c3c")
axes[1].set_title("Top 20 Junctions by Avg Severity")
axes[1].set_xlabel("Avg Severity Score")

plt.tight_layout()
plt.savefig("eda_09_junctions.png", bbox_inches="tight")
plt.show()
print("\nTop 10 junctions table:")
print(top_junctions.head(10))


In [ ]:
# ── 5j. Validation / Data quality analysis ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Validation status
val_counts = df["validation_status"].value_counts(dropna=False)
val_counts.index = val_counts.index.fillna("Unvalidated")
axes[0].pie(val_counts.values, labels=val_counts.index,
            autopct="%1.1f%%", colors=["#2ecc71","#e74c3c","#95a5a6"])
axes[0].set_title("Validation Status Distribution")

# SCITA flag
scita_counts = df["sent_to_scita"].value_counts()
scita_counts.index = ["Sent","Not Sent"]
axes[1].bar(scita_counts.index, scita_counts.values, color=["#3498db","#e74c3c"])
axes[1].set_title("Data Sent to SCITA")
axes[1].set_ylabel("Count")

# Num violations per record
num_viol_counts = df["num_violations"].value_counts().sort_index()
axes[2].bar(num_viol_counts.index.astype(str), num_viol_counts.values, color="#9b59b6")
axes[2].set_title("Number of Violations per Record")
axes[2].set_xlabel("Count of Violations Listed")
axes[2].set_ylabel("Records")

plt.tight_layout()
plt.savefig("eda_10_quality.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 5k. Geographic scatter (quick visual) ─────────────────────────────────
# Lightweight matplotlib scatter — not interactive, but fast for 3L rows

fig, ax = plt.subplots(figsize=(10, 10))
scatter = ax.scatter(
    df["longitude"], df["latitude"],
    c=df["severity_score"], cmap="YlOrRd",
    alpha=0.15, s=2, vmin=0, vmax=df["severity_score"].quantile(0.95)
)
plt.colorbar(scatter, ax=ax, label="Severity Score")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Geographic Distribution of Violations\n(coloured by severity)")
plt.tight_layout()
plt.savefig("eda_11_geo_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → eda_11_geo_scatter.png")


In [ ]:
# ── 5l. Interactive Folium heatmap (sample 10k for speed) ────────────────
from folium.plugins import HeatMap

sample = df.sample(min(10_000, len(df)), random_state=42)
heat_data = list(zip(sample["latitude"], sample["longitude"],
                     sample["severity_score"].fillna(1)))

m = folium.Map(location=[12.97, 77.59], zoom_start=12, tiles="CartoDB dark_matter")
HeatMap(heat_data, radius=10, blur=12, max_zoom=14,
        gradient={"0.2":"blue","0.5":"lime","0.8":"orange","1.0":"red"}).add_to(m)

folium.LayerControl().add_to(m)
m.save("violation_heatmap_day1.html")
print("✅ Interactive heatmap saved → violation_heatmap_day1.html")
print("Open this file in a browser to explore violations geographically.")
m


In [ ]:
# ── 5m. EDA Summary printout ──────────────────────────────────────────────
print("=" * 65)
print("EDA SUMMARY — KEY NUMBERS FOR DAY 2 MODEL")
print("=" * 65)
print(f"  Total clean records         : {len(df):,}")
print(f"  Date range                  : {df['date'].min()}  →  {df['date'].max()}")
print(f"  Unique vehicle types        : {df['vehicle_type'].nunique()}")
print(f"  Unique police stations      : {df['police_station'].nunique()}")
print(f"  Records near named junction : {df['near_junction'].sum():,} ({df['near_junction'].mean()*100:.1f}%)")
print(f"  Peak-hour records           : {df['is_peak_hour'].sum():,} ({df['is_peak_hour'].mean()*100:.1f}%)")
print(f"  Mean severity score         : {df['severity_score'].mean():.2f}")
print(f"  Mean impact score           : {df['impact_score'].mean():.2f}")
print(f"  Sent to SCITA               : {df['sent_to_scita'].mean()*100:.1f}%")
print(f"  Approved records            : {df['is_approved'].mean()*100:.1f}%")
print()
print("  Peak hour:    ", df.groupby("hour")["id"].count().idxmax(), ":00 IST")
print("  Worst day:    ", df.groupby("day_name")["id"].count().idxmax())
print("  Top station:  ", df["police_station"].value_counts().index[0])
print("  Top junction: ", df[df["junction_name"].notna()]["junction_name"].value_counts().index[0]
      if df["junction_name"].notna().any() else "N/A")
print("=" * 65)


---
## 6. Save Clean Data

Save as **Parquet** (10× faster to load than CSV, preserves dtypes).
Day 2 will load this directly — no re-cleaning needed.


In [ ]:
# ── 6a. Select columns to keep ────────────────────────────────────────────
KEEP_COLS = [
    "id", "latitude", "longitude", "location",
    "vehicle_number", "vehicle_type", "violation_type",
    "violation_list", "offence_code_list", "num_violations",
    "created_ist", "hour", "day_of_week", "day_name",
    "month", "month_name", "week", "date",
    "is_peak_hour", "is_morning_peak", "is_evening_peak",
    "is_weekend", "time_bucket",
    "severity_score", "vehicle_weight", "road_rank",
    "near_junction", "impact_score",
    "junction_name", "police_station", "center_code",
    "data_sent_to_scita", "sent_to_scita",
    "validation_status", "is_approved", "is_rejected", "is_validated",
]

df_clean = df[[c for c in KEEP_COLS if c in df.columns]].copy()
print(f"Saving {len(df_clean):,} rows × {df_clean.shape[1]} columns...")


In [ ]:
df_clean.to_parquet("violations_clean_day1.parquet", index=False)
print("✅ Saved → violations_clean_day1.parquet")
print(f"   File size: {Path('violations_clean_day1.parquet').stat().st_size / 1e6:.1f} MB")

# Also save a small CSV preview for quick inspection
df_clean.head(500).to_csv("violations_preview_500.csv", index=False)
print("✅ Saved preview → violations_preview_500.csv")


In [ ]:
# ── 6b. Final checklist ───────────────────────────────────────────────────
print("""
✅  DAY 1 COMPLETE — CHECKLIST
───────────────────────────────────────────────────────
[x] Data loaded and inspected
[x] Duplicates & invalid coordinates removed
[x] Datetime parsed and converted to IST
[x] violation_type and offence_code arrays parsed
[x] Temporal features: hour, day, month, peak flags
[x] Severity score computed (offence-code weighted)
[x] Vehicle weight (congestion footprint) assigned
[x] Road rank inferred from address string
[x] Junction proximity flagged
[x] Per-record impact score computed
[x] 11 EDA charts saved as PNG
[x] Interactive heatmap saved as HTML
[x] Clean data saved as Parquet

READY FOR DAY 2:
  → H3 hexagonal binning
  → Spatial clustering (DBSCAN)
  → Zone-level priority scoring
  → Temporal forecasting per zone
""")
